# Fitting

## Setup and configuration

In [ ]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams.update({
    "figure.figsize": (9, 6),
    "axes.grid": True,
})

COLUMNS = [
    "Freq_Hz",
    "Time_s",
    "Eps_real",
    "Eps_imag",
    "Temp_K",
    "MTime_s",
    "Phi_deg",
    "Z_real_Ohm",
    "Z_imag_Ohm",
    "TanPhi",
]

TIME_COL = "Time_Relative_s"

MATERIAL = "PEI5mgmL"
TEMPERATURE = "50°C"
MODE = "Abs"
FIT_TIME_S = 0

IGNORE_LAST_FREQ_POINTS = 3
FIT_EPS_IMAG = True

# Atmosphere switch points are maintained centrally in Code/switch_points.py.


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data" / "Daten final").exists():
            return path
    raise FileNotFoundError("Could not find the project root with data/Daten final.")




def add_code_dir_to_path(project_root):
    code_dir = project_root / "Code"
    code_dir_text = str(code_dir)
    if code_dir_text not in sys.path:
        sys.path.insert(0, code_dir_text)


PROJECT_ROOT = find_project_root()
add_code_dir_to_path(PROJECT_ROOT)
import switch_points
importlib.reload(switch_points)
manual_switch_points = switch_points.MANUAL_SWITCH_POINTS

DATA_DIR = PROJECT_ROOT / "data" / "Daten final"
SELECTED_FILE = f"{MATERIAL}_{TEMPERATURE}_{MODE}.TXT"
DATA_DIR / SELECTED_FILE

## Load data and set relative time

In [ ]:
def load_measurements(data_dir=DATA_DIR, selected_file=None, material_filter="PEI5mgmL"):
    if selected_file is None:
        file_paths = sorted(data_dir.glob("*.TXT")) + sorted(data_dir.glob("*.txt"))
    else:
        file_path = data_dir / selected_file
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
        file_paths = [file_path]

    if selected_file is None and material_filter is not None:
        file_paths = [
            path for path in file_paths
            if path.stem.split("_")[0] == material_filter
        ]

    if not file_paths:
        raise FileNotFoundError(f"No .TXT files found in: {data_dir}")

    frames = []
    for path in file_paths:
        df = pd.read_csv(
            path,
            sep=r"\s+",
            skiprows=4,
            names=COLUMNS,
            encoding="latin1",
            engine="python",
        )

        parts = path.stem.split("_")
        material, temperature, mode = parts[:3] if len(parts) >= 3 else (path.stem, None, None)
        first_freq = df["Freq_Hz"].iloc[0]
        df["Spectrum_ID"] = (df["Freq_Hz"] == first_freq).cumsum() - 1
        df["Spectrum_Number"] = df["Spectrum_ID"] + 1
        df["Point_Number"] = range(1, len(df) + 1)
        df["Material"] = material
        df["Temperature"] = temperature
        df["Mode"] = mode
        df["Source_File"] = path.name
        frames.append(df)

    return pd.concat(frames, ignore_index=True)


def add_relative_switch_time(df, switch_points):
    df = df.copy()
    switch_rows = []

    for dataset_key, dataset in df.groupby(["Material", "Temperature", "Mode"], dropna=False):
        switch_point_number = switch_points.get(dataset_key)
        if switch_point_number is None:
            raise ValueError(f"Missing Switch_Point_Number for series: {dataset_key}")

        switch_point = dataset[dataset["Point_Number"] == switch_point_number]
        next_point = dataset[dataset["Point_Number"] == switch_point_number + 1]
        if switch_point.empty or next_point.empty:
            raise ValueError(f"Switch-point pair not found: {dataset_key}, {switch_point_number}")

        switch_time = float((switch_point["MTime_s"].iloc[0] + next_point["MTime_s"].iloc[0]) / 2)
        switch_rows.append({
            "Material": dataset_key[0],
            "Temperature": dataset_key[1],
            "Mode": dataset_key[2],
            "Switch_Point_Number": int(switch_point_number),
            "Switch_Time_s": switch_time,
        })

    switch_times = pd.DataFrame(switch_rows)
    df = df.merge(switch_times, on=["Material", "Temperature", "Mode"], how="left")
    df[TIME_COL] = df["MTime_s"] - df["Switch_Time_s"]
    return df, switch_times

In [ ]:
df_raw = load_measurements(selected_file=SELECTED_FILE)
df_raw, switch_times = add_relative_switch_time(df_raw, manual_switch_points)

print(f"Loaded rows: {len(df_raw):,}")
print(switch_times.to_string(index=False))

## Interpolation and derivative data

In [ ]:
def _linear_at(target_time, times, values):
    if len(times) < 2:
        return np.nan

    x0, x1 = float(times[0]), float(times[1])
    y0, y1 = float(values[0]), float(values[1])

    if x0 == x1:
        return np.nan

    return y0 + (target_time - x0) * (y1 - y0) / (x1 - x0)


def _insert_switch_support_point(freq_data, value_col, time_col=TIME_COL, target_time=0):
    before = freq_data[freq_data[time_col] < target_time].tail(2)
    support_points = freq_data[[time_col, value_col]].dropna().copy()
    support_points = support_points[~np.isclose(support_points[time_col], target_time)]

    if len(before) >= 2:
        switch_value = _linear_at(target_time, before[time_col].to_numpy(), before[value_col].to_numpy())
        support_points = pd.concat(
            [support_points, pd.DataFrame({time_col: [float(target_time)], value_col: [switch_value]})],
            ignore_index=True,
        )

    support_points = support_points.sort_values(time_col)
    return support_points.groupby(time_col, as_index=False)[value_col].mean()


def interpolate_complete_spectra(
    df,
    time_col=TIME_COL,
    freq_col="Freq_Hz",
    value_cols=("Eps_real", "Eps_imag"),
    dataset_cols=("Source_File", "Material", "Temperature", "Mode"),
    drop_incomplete=True,
):
    rows = []

    for dataset_key, dataset in df.groupby(list(dataset_cols), dropna=False):
        dataset_key = dataset_key if isinstance(dataset_key, tuple) else (dataset_key,)
        dataset_meta = dict(zip(dataset_cols, dataset_key))
        target_times = np.sort(dataset[time_col].dropna().unique())
        if not np.isclose(target_times, 0).any():
            target_times = np.sort(np.append(target_times, 0.0))
        frequencies = np.sort(dataset[freq_col].dropna().unique())

        interpolated_by_freq = {}
        for freq in frequencies:
            freq_data = dataset.loc[dataset[freq_col] == freq, [time_col, *value_cols]].dropna(subset=[time_col])
            freq_data = freq_data.sort_values(time_col).groupby(time_col, as_index=False)[list(value_cols)].mean()

            interpolated_values = {}
            for value_col in value_cols:
                support = _insert_switch_support_point(freq_data, value_col, time_col=time_col, target_time=0)
                interpolated_values[value_col] = np.interp(
                    target_times,
                    support[time_col].to_numpy(),
                    support[value_col].to_numpy(),
                    left=np.nan,
                    right=np.nan,
                )
            interpolated_by_freq[freq] = interpolated_values

        for spectrum_id, target_time in enumerate(target_times):
            for freq in frequencies:
                row = {
                    **dataset_meta,
                    "Interpolated_Spectrum_ID": spectrum_id,
                    time_col: target_time,
                    freq_col: freq,
                    "Is_Switch_Spectrum": bool(np.isclose(target_time, 0)),
                }
                for value_col in value_cols:
                    row[value_col] = interpolated_by_freq[freq][value_col][spectrum_id]
                rows.append(row)

    interpolated = pd.DataFrame(rows)

    if drop_incomplete:
        complete_ids = [*dataset_cols, "Interpolated_Spectrum_ID"]
        complete_mask = interpolated.groupby(complete_ids, dropna=False)[list(value_cols)].transform(
            lambda values: values.notna().all()
        )
        interpolated = interpolated[complete_mask.all(axis=1)].reset_index(drop=True)

    return interpolated


def derive_eps_real_by_frequency(
    df,
    value_col="Eps_real",
    freq_col="Freq_Hz",
    time_col=TIME_COL,
    ignored_last_points=IGNORE_LAST_FREQ_POINTS,
):
    rows = []
    group_cols = ["Source_File", "Material", "Temperature", "Mode", "Interpolated_Spectrum_ID", time_col]

    for group_key, spectrum in df.groupby(group_cols, dropna=False):
        spectrum = spectrum.sort_values(freq_col).reset_index(drop=True)
        if ignored_last_points:
            spectrum = spectrum.iloc[:-ignored_last_points]
        if len(spectrum) < 2:
            continue

        frequencies = spectrum[freq_col].to_numpy()
        eps_real = spectrum[value_col].to_numpy()
        omega = 2 * np.pi * frequencies
        ln_omega = np.log(omega)
        derivative = -np.pi / 2 * np.diff(eps_real) / np.diff(ln_omega)
        omega_mid = np.exp((ln_omega[:-1] + ln_omega[1:]) / 2)
        freq_mid = omega_mid / (2 * np.pi)
        group_key = group_key if isinstance(group_key, tuple) else (group_key,)
        meta = dict(zip(group_cols, group_key))

        for freq, omega_value, derivative_value in zip(freq_mid, omega_mid, derivative):
            rows.append({
                **meta,
                "Freq_Hz_mid": freq,
                "Omega_rad_s": omega_value,
                "Eps_real_derivative": derivative_value,
            })

    return pd.DataFrame(rows)

In [ ]:
df_interpolated = interpolate_complete_spectra(df_raw)
df_derivative = derive_eps_real_by_frequency(df_interpolated)

print(f"Interpolated rows: {len(df_interpolated):,}")
print(f"Derivative rows: {len(df_derivative):,}")
df_derivative.head()

## Cole-Cole model

In [ ]:
TINY = 1e-30


def cc_imag(w, de, alpha, wp):
    den = 1 + (1j * w / wp) ** alpha
    return -np.imag(de / den)


def cc_real_derivative(w, de, alpha, wp):
    A = alpha * np.pi / 2
    W = (w / wp) ** alpha
    return (
        A
        * de
        * W
        * np.cos(A - 2 * np.arctan(np.sin(A) / (1 / W + np.cos(A))))
        / (1 + 2 * W * np.cos(A) + W**2)
    )


def sum_terms(w, params, term_function):
    y = np.zeros_like(w, dtype=float)
    for de, alpha, wp in np.array(params).reshape(-1, 3):
        y += term_function(w, de, alpha, wp)
    return y


def derivative_log_model(lnw, *params):
    w = np.exp(lnw)
    return np.log(np.maximum(sum_terms(w, params, cc_real_derivative), TINY))


def imag_log_model(lnw, *params):
    w = np.exp(lnw)
    return np.log(np.maximum(sum_terms(w, params, cc_imag), TINY))


def combined_log_model(x_all, *params):
    lnw, mask = x_all
    return np.where(mask == 0, derivative_log_model(lnw, *params), imag_log_model(lnw, *params))

# Parameters per term: delta epsilon, alpha, omega_peak [rad/s].
P0 = np.array([
    1.0, 0.7, 1e2,
    0.5, 0.5, 1e5,
], dtype=float)

LOWER_BOUNDS = np.array([
    1e-10, 0.05, 1e-8,
    1e-10, 0.05, 1e1,
], dtype=float)

UPPER_BOUNDS = np.array([
    np.inf, 1.0, 1e4,
    np.inf, 1.0, 1e9,
], dtype=float)

PARAMETER_LABELS = [
    "de_1", "alpha_1", "omega_p_1",
    "de_2", "alpha_2", "omega_p_2",
]

N_COLE_TERMS = len(P0) // 3

## Fit one time point

In [ ]:
def clip_to_bounds(p0):
    return np.minimum(np.maximum(np.asarray(p0, dtype=float), LOWER_BOUNDS * 1.000001), UPPER_BOUNDS * 0.999999)


def fit_quality(y_data, y_fit, popt):
    residual = y_data - y_fit
    rmse_log = float(np.sqrt(np.nanmean(residual**2)))
    ss_res = float(np.nansum(residual**2))
    ss_tot = float(np.nansum((y_data - np.nanmean(y_data)) ** 2))
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan
    lower_hit = np.isclose(popt, LOWER_BOUNDS, rtol=1e-4, atol=1e-12)
    upper_hit = np.isclose(popt, UPPER_BOUNDS, rtol=1e-4, atol=1e-12)
    at_bounds = bool(np.any(lower_hit | upper_hit))
    return {
        "RMSE_Log": rmse_log,
        "R2": r2,
        "At_Bounds": at_bounds,
        "Bounds_Hit_Count": int(np.sum(lower_hit | upper_hit)),
    }


def is_fit_accepted(quality, max_rmse_log=1.0, min_r2=0.5, allow_bounds=False):
    if not np.isfinite(quality["RMSE_Log"]):
        return False
    if quality["RMSE_Log"] > max_rmse_log:
        return False
    if np.isfinite(quality["R2"]) and quality["R2"] < min_r2:
        return False
    if quality["At_Bounds"] and not allow_bounds:
        return False
    return True


def build_fit_data(fit_time_s, fit_eps_imag=FIT_EPS_IMAG):
    available_times = np.sort(df_derivative[TIME_COL].unique())
    plot_time = available_times[np.abs(available_times - fit_time_s).argmin()]

    derivative_df = df_derivative[df_derivative[TIME_COL] == plot_time].sort_values("Omega_rad_s")
    derivative_df = derivative_df[(derivative_df["Eps_real_derivative"] > 0) & np.isfinite(derivative_df["Eps_real_derivative"])]
    x_der = np.log(derivative_df["Omega_rad_s"].to_numpy())
    y_der = np.log(derivative_df["Eps_real_derivative"].to_numpy())

    if fit_eps_imag:
        imag_df = df_interpolated[df_interpolated[TIME_COL] == plot_time].sort_values("Freq_Hz")
        if IGNORE_LAST_FREQ_POINTS:
            imag_df = imag_df.iloc[:-IGNORE_LAST_FREQ_POINTS]
        imag_df = imag_df[(imag_df["Eps_imag"] > 0) & np.isfinite(imag_df["Eps_imag"])]
        x_imag = np.log(2 * np.pi * imag_df["Freq_Hz"].to_numpy())
        y_imag = np.log(imag_df["Eps_imag"].to_numpy())
        x_all = np.concatenate([x_der, x_imag])
        y_all = np.concatenate([y_der, y_imag])
        mask = np.concatenate([np.zeros_like(x_der), np.ones_like(x_imag)])
        model = combined_log_model
        x_fit = (x_all, mask)
    else:
        y_all = y_der
        model = derivative_log_model
        x_fit = x_der

    return plot_time, x_fit, y_all, model, derivative_df


def fit_spectrum_at_time(
    fit_time_s,
    p0=P0,
    fit_eps_imag=FIT_EPS_IMAG,
    max_rmse_log=1.0,
    min_r2=0.5,
    allow_bounds=False,
):
    plot_time, x_fit, y_data, model, derivative_df = build_fit_data(fit_time_s, fit_eps_imag=fit_eps_imag)
    p0 = clip_to_bounds(p0)

    popt, pcov = curve_fit(
        model,
        x_fit,
        y_data,
        p0=p0,
        bounds=(LOWER_BOUNDS, UPPER_BOUNDS),
        maxfev=50000,
    )

    quality = fit_quality(y_data, model(x_fit, *popt), popt)
    quality["Accepted"] = is_fit_accepted(
        quality,
        max_rmse_log=max_rmse_log,
        min_r2=min_r2,
        allow_bounds=allow_bounds,
    )
    return plot_time, popt, pcov, derivative_df, quality

In [ ]:
fit_time, popt, pcov, fit_derivative_df, fit_quality_result = fit_spectrum_at_time(FIT_TIME_S)
fit_result = pd.DataFrame({"parameter": PARAMETER_LABELS, "value": popt})

print(f"Fit time used: {fit_time:.2f} s")
print(fit_quality_result)
fit_result

## Plot helpers

In [ ]:
def plot_single_fit(fit_time, popt, derivative_df):
    w_plot = np.logspace(
        np.log10(derivative_df["Omega_rad_s"].min()),
        np.log10(derivative_df["Omega_rad_s"].max()),
        400,
    )

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(
        derivative_df["Omega_rad_s"],
        derivative_df["Eps_real_derivative"],
        color="black",
        label="Interpolated-spectrum derivative",
    )

    fit_total = sum_terms(w_plot, popt, cc_real_derivative)
    ax.plot(w_plot, fit_total, color="red", linewidth=2.2, label="Total Cole-Cole fit")

    for term_index, term_params in enumerate(np.array(popt).reshape(-1, 3), start=1):
        term_fit = cc_real_derivative(w_plot, *term_params)
        ax.plot(
            w_plot,
            term_fit,
            linestyle="--",
            linewidth=1.4,
            label=f"Cole-Cole term {term_index}",
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Angular frequency omega (rad/s)")
    ax.set_ylabel("Eps real derivative")
    ax.set_title(f"Fit at t_rel = {fit_time:.2f} s | {MATERIAL} {TEMPERATURE} {MODE}")
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_fit_parameter_time_series(
    df_fit_parameters,
    parameters=("de_1", "de_2", "omega_p_1", "omega_p_2", "alpha_1", "alpha_2"),
    plot_only_accepted=True,
):
    if df_fit_parameters.empty:
        print("No time-series fit is available. Run the time-series fit with RUN_TIME_SERIES_FIT = True first.")
        return

    plot_params = df_fit_parameters[df_fit_parameters["Fit_Success"]].copy()
    if plot_only_accepted:
        plot_params = plot_params[plot_params["Accepted"]]

    print(
        df_fit_parameters.groupby(["Fit_Success", "Accepted"], dropna=False)
        .size()
        .rename("count")
        .reset_index()
    )

    for parameter in parameters:
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(plot_params["Time_Relative_s"], plot_params[parameter], marker="o")
        if parameter.startswith("omega") or parameter.startswith("de"):
            ax.set_yscale("log")
        ax.axvline(0, color="black", linestyle="--", linewidth=1)
        ax.set_xlabel("Time relative to atmosphere switch (s)")
        ax.set_ylabel(parameter)
        ax.set_title(f"Parameter over time: {parameter} | {MATERIAL} {TEMPERATURE} {MODE}")
        plt.tight_layout()
        plt.show()

    quality_cols = ["Time_Relative_s", "Direction", "Fit_Success", "Accepted", "RMSE_Log", "R2", "At_Bounds", "Bounds_Hit_Count"]
    print(df_fit_parameters[quality_cols].head(20))

In [ ]:
plot_single_fit(fit_time, popt, fit_derivative_df)

## Time-series fit

In [ ]:
RUN_TIME_SERIES_FIT = False

FIT_TIME_MIN_S = -300
FIT_TIME_MAX_S = 900
FIT_EVERY_N = 5
FIT_REFERENCE_TIME_S = 0

MAX_RMSE_LOG = 1.0
MIN_R2 = 0.5
ALLOW_BOUND_HITS = False


def fit_time_sequence(times, start_p0, direction_label):
    rows = []
    p0_current = start_p0.copy()

    for time_s in times:
        try:
            actual_time, popt_i, _, _, quality = fit_spectrum_at_time(
                time_s,
                p0=p0_current,
                max_rmse_log=MAX_RMSE_LOG,
                min_r2=MIN_R2,
                allow_bounds=ALLOW_BOUND_HITS,
            )
            row = {
                "Time_Relative_s": actual_time,
                "Fit_Success": True,
                "Direction": direction_label,
                **quality,
            }
            row.update(dict(zip(PARAMETER_LABELS, popt_i)))

            if quality["Accepted"]:
                p0_current = popt_i
        except Exception as err:
            row = {
                "Time_Relative_s": time_s,
                "Fit_Success": False,
                "Accepted": False,
                "Direction": direction_label,
                "Error": str(err),
            }
        rows.append(row)

    return rows


if RUN_TIME_SERIES_FIT:
    all_times = np.sort(df_derivative[TIME_COL].unique())
    all_times = all_times[(all_times >= FIT_TIME_MIN_S) & (all_times <= FIT_TIME_MAX_S)]
    all_times = all_times[::FIT_EVERY_N]

    if len(all_times) == 0:
        raise ValueError("No time points found in the selected fit window.")

    reference_time = all_times[np.abs(all_times - FIT_REFERENCE_TIME_S).argmin()]
    reference_time, reference_popt, _, _, reference_quality = fit_spectrum_at_time(
        reference_time,
        p0=P0,
        max_rmse_log=MAX_RMSE_LOG,
        min_r2=MIN_R2,
        allow_bounds=ALLOW_BOUND_HITS,
    )

    rows = []
    reference_row = {
        "Time_Relative_s": reference_time,
        "Fit_Success": True,
        "Direction": "reference",
        **reference_quality,
    }
    reference_row.update(dict(zip(PARAMETER_LABELS, reference_popt)))
    rows.append(reference_row)

    future_times = all_times[all_times > reference_time]
    past_times = all_times[all_times < reference_time][::-1]

    rows.extend(fit_time_sequence(future_times, reference_popt, "forward"))
    rows.extend(fit_time_sequence(past_times, reference_popt, "backward"))

    df_fit_parameters = (
        pd.DataFrame(rows)
        .sort_values("Time_Relative_s")
        .reset_index(drop=True)
    )

    print(df_fit_parameters.head())
else:
    df_fit_parameters = pd.DataFrame()
    print("Time-series fit is disabled. Set RUN_TIME_SERIES_FIT = True to fit all selected time points.")

In [ ]:
plot_fit_parameter_time_series(
    df_fit_parameters,
    parameters=("de_1", "de_2", "omega_p_1", "omega_p_2", "alpha_1", "alpha_2"),
    plot_only_accepted=True,
)